# ============================================================
# CAPSTONE PROJECT
# NOTEBOOK 40: FULL-161 RARE-TAIL FALLBACK ROUTER
# ============================================================
# Purpose:
# This notebook builds the Stage-2 fallback mechanism needed to
# move from the completed candidate-150 system toward a full
# 161-label pipeline.
#
# The goal is to:
# 1. Load the rare-tail audit outputs
# 2. Map each rare-tail label to the nearest candidate ancestor
# 3. Compute training co-occurrence statistics for fallback routing
# 4. Build anchor-to-rare-tail routing tables
# 5. Estimate routing coverage using ground-truth hierarchy
# 6. Save the Stage-2 fallback artifacts for the final full-genre pipeline
# ============================================================

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import os
import math
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

print("Seed set to:", SEED)

Seed set to: 42


In [2]:
# ============================================================
# 2. LOAD CORE TABLES
# ============================================================

genre_inventory_df = pd.read_csv("../data/processed/full_genre_inventory.csv")
strategy_df = pd.read_csv("../data/processed/full161_all_genre_strategy_table.csv")
rare_tail_summary_df = pd.read_csv("../data/processed/full161_rare_tail_summary.csv")
rare_tail_pairs_df = pd.read_csv("../data/processed/full161_rare_tail_track_label_pairs.csv")
full_master_df = pd.read_csv("../data/processed/multilabel_full_master_table.csv")

print("Genre inventory shape:", genre_inventory_df.shape)
print("Strategy table shape:", strategy_df.shape)
print("Rare-tail summary shape:", rare_tail_summary_df.shape)
print("Rare-tail track-label pairs shape:", rare_tail_pairs_df.shape)
print("Full master table shape:", full_master_df.shape)

display(genre_inventory_df.head())
display(strategy_df.head())
display(rare_tail_summary_df.head())
display(rare_tail_pairs_df.head())
display(full_master_df.head())

Genre inventory shape: (163, 11)
Strategy table shape: (163, 18)
Rare-tail summary shape: (13, 11)
Rare-tail track-label pairs shape: (124, 13)
Full master table shape: (81574, 170)


,genre_id,genre_name,parent_id,parent_name,root_genre_id,root_genre_name,top_level_flag,full_direct_count,full_genres_all_count,large_audio_genres_all_count,modelling_tier
0,38,Experimental,NaN,NaN,38,Experimental,38,24912,38154,35903,Tier 1: Strong
1,15,Electronic,NaN,NaN,15,Electronic,15,23866,34413,28099,Tier 1: Strong
2,12,Rock,NaN,NaN,12,Rock,12,8038,32923,25820,Tier 1: Strong
3,1235,Instrumental,NaN,NaN,1235,Instrumental,1235,6055,14938,13588,Tier 1: Strong
4,10,Pop,NaN,NaN,10,Pop,10,6362,13845,12659,Tier 1: Strong


,genre_id,genre_name,parent_id,parent_name,root_genre_id,root_genre_name,top_level_flag,full_direct_count,full_genres_all_count,large_audio_genres_all_count,modelling_tier,total_count,training_count,validation_count,test_count,feasibility_status,stage_role,recommended_action
0,38,Experimental,NaN,NaN,38,Experimental,38,24912,38154,35903,Tier 1: Strong,35903,NaN,NaN,NaN,Direct candidate label,Stage 1,Direct modelling in candidate-150 system
1,15,Electronic,NaN,NaN,15,Electronic,15,23866,34413,28099,Tier 1: Strong,28099,NaN,NaN,NaN,Direct candidate label,Stage 1,Direct modelling in candidate-150 system
2,12,Rock,NaN,NaN,12,Rock,12,8038,32923,25820,Tier 1: Strong,25820,NaN,NaN,NaN,Direct candidate label,Stage 1,Direct modelling in candidate-150 system
3,1235,Instrumental,NaN,NaN,1235,Instrumental,1235,6055,14938,13588,Tier 1: Strong,13588,NaN,NaN,NaN,Direct candidate label,Stage 1,Direct modelling in candidate-150 system
4,10,Pop,NaN,NaN,10,Pop,10,6362,13845,12659,Tier 1: Strong,12659,NaN,NaN,NaN,Direct candidate label,Stage 1,Direct modelling in candidate-150 system


,genre_id,genre_name,parent_id,parent_name,root_genre_id,root_genre_name,total_count,training_count,validation_count,test_count,feasibility_status
0,176,Pacific,2.0,International,2,International,23,17,2,4,Very scarce
1,1060,Tango,46.0,Latin America,2,International,23,5,6,12,Extremely scarce
2,465,Musical Theater,20.0,Spoken,20,Spoken,18,4,4,10,Extremely scarce
3,189,Talk Radio,65.0,Radio,20,Spoken,15,13,1,1,Very scarce
4,1032,Turkish,102.0,Middle East,2,International,15,10,0,5,Partial split support


,track_id,split,subset,genre_top,title,audio_path,audio_exists,rare_tail_genre_id,rare_tail_genre_name,parent_id,parent_name,root_genre_id,root_genre_name
0,13077,validation,large,NaN,Thanam-Kalyani,../data/raw/audio/fma_large\013\013077.mp3,True,174,South Indian Traditional,86.0,Indian,2,International
1,16930,test,large,NaN,Open Session,../data/raw/audio/fma_large\016\016930.mp3,True,189,Talk Radio,65.0,Radio,20,Spoken
2,19777,validation,large,NaN,"Custer: ""If I Were An Indian...""",../data/raw/audio/fma_large\019\019777.mp3,True,465,Musical Theater,20.0,Spoken,20,Spoken
3,19778,validation,large,NaN,Custer's Ghost To Sitting Bull,../data/raw/audio/fma_large\019\019778.mp3,True,465,Musical Theater,20.0,Spoken,20,Spoken
4,19779,validation,large,NaN,"Sitting Bull: ""Do You Know Who I Am?""",../data/raw/audio/fma_large\019\019779.mp3,True,465,Musical Theater,20.0,Spoken,20,Spoken


,track_id,split,subset,genre_top,title,audio_path,audio_exists,genre_1,genre_2,genre_3,...,genre_763,genre_808,genre_810,genre_811,genre_906,genre_1032,genre_1060,genre_1156,genre_1193,genre_1235
0,20,training,large,NaN,Spiritual Level,../data/raw/audio/fma_large\000\000020.mp3,True,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,26,training,large,NaN,Where is your Love?,../data/raw/audio/fma_large\000\000026.mp3,True,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,30,training,large,NaN,Too Happy,../data/raw/audio/fma_large\000\000030.mp3,True,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,46,training,large,NaN,Yosemite,../data/raw/audio/fma_large\000\000046.mp3,True,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,48,training,large,NaN,Light of Light,../data/raw/audio/fma_large\000\000048.mp3,True,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [3]:
# ============================================================
# 3. PREPARE LOOKUP STRUCTURES
# ============================================================

genre_inventory_df["genre_id"] = genre_inventory_df["genre_id"].astype(int)

for col in ["parent_id", "root_genre_id", "top_level_flag"]:
    if col in genre_inventory_df.columns:
        genre_inventory_df[col] = pd.to_numeric(genre_inventory_df[col], errors="coerce")

genre_name_map = dict(zip(genre_inventory_df["genre_id"], genre_inventory_df["genre_name"]))
parent_id_map = dict(zip(genre_inventory_df["genre_id"], genre_inventory_df["parent_id"]))
parent_name_map = dict(zip(genre_inventory_df["genre_id"], genre_inventory_df["parent_name"]))
root_id_map = dict(zip(genre_inventory_df["genre_id"], genre_inventory_df["root_genre_id"]))
root_name_map = dict(zip(genre_inventory_df["genre_id"], genre_inventory_df["root_genre_name"]))

candidate_genre_ids = set(
    strategy_df.loc[strategy_df["stage_role"] == "Stage 1", "genre_id"].astype(int).tolist()
)

rare_tail_genre_ids = set(
    strategy_df.loc[strategy_df["stage_role"] == "Stage 2", "genre_id"].astype(int).tolist()
)

print("Candidate genre count:", len(candidate_genre_ids))
print("Rare-tail genre count:", len(rare_tail_genre_ids))

Candidate genre count: 150
Rare-tail genre count: 13


In [4]:
# ============================================================
# 4. IDENTIFY LABEL COLUMNS
# ============================================================

label_cols = [
    col for col in full_master_df.columns
    if col.startswith("genre_") and col.replace("genre_", "").isdigit()
]

label_col_ids = {int(col.replace("genre_", "")) for col in label_cols}

candidate_label_cols = [
    f"genre_{gid}" for gid in sorted(candidate_genre_ids) if f"genre_{gid}" in full_master_df.columns
]

rare_tail_label_cols = [
    f"genre_{gid}" for gid in sorted(rare_tail_genre_ids) if f"genre_{gid}" in full_master_df.columns
]

print("Total label columns in full master:", len(label_cols))
print("Candidate label columns found:", len(candidate_label_cols))
print("Rare-tail label columns found:", len(rare_tail_label_cols))

Total label columns in full master: 163
Candidate label columns found: 150
Rare-tail label columns found: 13


In [5]:
# ============================================================
# 5. HIERARCHY HELPER FUNCTIONS
# ============================================================

def normalize_int_or_none(x):
    if pd.isna(x):
        return None
    try:
        return int(x)
    except Exception:
        return None

def get_ancestor_chain(genre_id, parent_map):
    """
    Returns ancestor chain from immediate parent upward.
    Example: Tango -> Latin America -> International
    """
    chain = []
    visited = set()

    current = normalize_int_or_none(parent_map.get(genre_id, None))

    while current is not None and current not in visited:
        chain.append(current)
        visited.add(current)
        current = normalize_int_or_none(parent_map.get(current, None))

    return chain

def get_nearest_candidate_ancestor(genre_id, parent_map, candidate_set):
    chain = get_ancestor_chain(genre_id, parent_map)
    for anc in chain:
        if anc in candidate_set:
            return anc
    return None

In [6]:
# ============================================================
# 6. BUILD RARE-TAIL ROUTER TABLE
# ============================================================

router_rows = []

for _, row in rare_tail_summary_df.iterrows():
    gid = int(row["genre_id"])
    gname = row["genre_name"]

    ancestor_chain_ids = get_ancestor_chain(gid, parent_id_map)
    ancestor_chain_names = [genre_name_map.get(x) for x in ancestor_chain_ids]

    anchor_candidate_id = get_nearest_candidate_ancestor(gid, parent_id_map, candidate_genre_ids)
    anchor_candidate_name = genre_name_map.get(anchor_candidate_id) if anchor_candidate_id is not None else None

    root_candidate_id = normalize_int_or_none(root_id_map.get(gid, None))
    root_candidate_name = genre_name_map.get(root_candidate_id) if root_candidate_id is not None else None

    if row["feasibility_status"] == "No train support":
        fallback_mode = "Inventory only"
    elif row["feasibility_status"] in ["Train only", "Partial split support", "Extremely scarce", "Very scarce"]:
        fallback_mode = "Hierarchy-triggered fallback"
    else:
        fallback_mode = "Review"

    router_rows.append({
        "rare_tail_genre_id": gid,
        "rare_tail_genre_name": gname,
        "parent_id": normalize_int_or_none(row["parent_id"]),
        "parent_name": row["parent_name"],
        "root_genre_id": normalize_int_or_none(row["root_genre_id"]),
        "root_genre_name": row["root_genre_name"],
        "training_count": int(row["training_count"]),
        "validation_count": int(row["validation_count"]),
        "test_count": int(row["test_count"]),
        "total_count": int(row["total_count"]),
        "feasibility_status": row["feasibility_status"],
        "ancestor_chain_ids": " > ".join([str(x) for x in ancestor_chain_ids]),
        "ancestor_chain_names": " > ".join([str(x) for x in ancestor_chain_names]),
        "anchor_candidate_id": anchor_candidate_id,
        "anchor_candidate_name": anchor_candidate_name,
        "root_candidate_id": root_candidate_id,
        "root_candidate_name": root_candidate_name,
        "fallback_mode": fallback_mode
    })

rare_tail_router_df = pd.DataFrame(router_rows).sort_values(
    ["fallback_mode", "total_count", "training_count"],
    ascending=[True, False, False]
).reset_index(drop=True)

print("Rare-tail router table:")
display(rare_tail_router_df)

Rare-tail router table:


,rare_tail_genre_id,rare_tail_genre_name,parent_id,parent_name,root_genre_id,root_genre_name,training_count,validation_count,test_count,total_count,feasibility_status,ancestor_chain_ids,ancestor_chain_names,anchor_candidate_id,anchor_candidate_name,root_candidate_id,root_candidate_name,fallback_mode
0,176,Pacific,2,International,2,International,17,2,4,23,Very scarce,2,International,2,International,2,International,Hierarchy-triggered fallback
1,1060,Tango,46,Latin America,2,International,5,6,12,23,Extremely scarce,46 > 2,Latin America > International,46,Latin America,2,International,Hierarchy-triggered fallback
2,465,Musical Theater,20,Spoken,20,Spoken,4,4,10,18,Extremely scarce,20,Spoken,20,Spoken,20,Spoken,Hierarchy-triggered fallback
3,189,Talk Radio,65,Radio,20,Spoken,13,1,1,15,Very scarce,65 > 20,Radio > Spoken,65,Radio,20,Spoken,Hierarchy-triggered fallback
4,1032,Turkish,102,Middle East,2,International,10,0,5,15,Partial split support,102 > 2,Middle East > International,102,Middle East,2,International,Hierarchy-triggered fallback
5,374,Banter,20,Spoken,20,Spoken,5,0,0,5,Train only,20,Spoken,20,Spoken,20,Spoken,Hierarchy-triggered fallback
6,173,N. Indian Traditional,86,Indian,2,International,3,0,1,4,Partial split support,86 > 2,Indian > International,86,Indian,2,International,Hierarchy-triggered fallback
7,493,Western Swing,651,Country & Western,9,Country,1,1,2,4,Extremely scarce,651 > 9,Country & Western > Country,651,Country & Western,9,Country,Hierarchy-triggered fallback
8,377,Deep Funk,19,Funk,14,Soul-RnB,1,0,0,1,Train only,19 > 14,Funk > Soul-RnB,19,Funk,14,Soul-RnB,Hierarchy-triggered fallback
9,808,Salsa,46,Latin America,2,International,1,0,0,1,Train only,46 > 2,Latin America > International,46,Latin America,2,International,Hierarchy-triggered fallback


In [7]:
# ============================================================
# 7. COMPUTE TRAINING CO-OCCURRENCE STATISTICS
# ============================================================

train_df = full_master_df[full_master_df["split"] == "training"].copy().reset_index(drop=True)

cooccurrence_rows = []

for _, row in rare_tail_router_df.iterrows():
    rare_id = int(row["rare_tail_genre_id"])
    rare_col = f"genre_{rare_id}"

    anchor_id = row["anchor_candidate_id"]
    root_id = row["root_candidate_id"]

    rare_train_count = int(train_df[rare_col].sum()) if rare_col in train_df.columns else 0

    # Anchor stats
    if anchor_id is not None and f"genre_{anchor_id}" in train_df.columns:
        anchor_col = f"genre_{anchor_id}"
        anchor_train_count = int(train_df[anchor_col].sum())
        cooccur_anchor_count = int(((train_df[rare_col] == 1) & (train_df[anchor_col] == 1)).sum())
        p_rare_given_anchor = cooccur_anchor_count / anchor_train_count if anchor_train_count > 0 else np.nan
        p_anchor_given_rare = cooccur_anchor_count / rare_train_count if rare_train_count > 0 else np.nan
    else:
        anchor_train_count = 0
        cooccur_anchor_count = 0
        p_rare_given_anchor = np.nan
        p_anchor_given_rare = np.nan

    # Root stats
    if root_id is not None and f"genre_{root_id}" in train_df.columns:
        root_col = f"genre_{root_id}"
        root_train_count = int(train_df[root_col].sum())
        cooccur_root_count = int(((train_df[rare_col] == 1) & (train_df[root_col] == 1)).sum())
        p_rare_given_root = cooccur_root_count / root_train_count if root_train_count > 0 else np.nan
        p_root_given_rare = cooccur_root_count / rare_train_count if rare_train_count > 0 else np.nan
    else:
        root_train_count = 0
        cooccur_root_count = 0
        p_rare_given_root = np.nan
        p_root_given_rare = np.nan

    cooccurrence_rows.append({
        "rare_tail_genre_id": rare_id,
        "rare_tail_genre_name": row["rare_tail_genre_name"],
        "training_count": rare_train_count,
        "anchor_candidate_id": anchor_id,
        "anchor_candidate_name": row["anchor_candidate_name"],
        "anchor_train_count": anchor_train_count,
        "cooccur_anchor_count": cooccur_anchor_count,
        "p_rare_given_anchor": p_rare_given_anchor,
        "p_anchor_given_rare": p_anchor_given_rare,
        "root_candidate_id": root_id,
        "root_candidate_name": row["root_candidate_name"],
        "root_train_count": root_train_count,
        "cooccur_root_count": cooccur_root_count,
        "p_rare_given_root": p_rare_given_root,
        "p_root_given_rare": p_root_given_rare
    })

cooccurrence_df = pd.DataFrame(cooccurrence_rows)

print("Rare-tail co-occurrence table:")
display(cooccurrence_df)

Rare-tail co-occurrence table:


,rare_tail_genre_id,rare_tail_genre_name,training_count,anchor_candidate_id,anchor_candidate_name,anchor_train_count,cooccur_anchor_count,p_rare_given_anchor,p_anchor_given_rare,root_candidate_id,root_candidate_name,root_train_count,cooccur_root_count,p_rare_given_root,p_root_given_rare
0,176,Pacific,17,2,International,3311,17,0.005134,1.0,2,International,3311,17,0.005134,1.0
1,1060,Tango,5,46,Latin America,351,5,0.014245,1.0,2,International,3311,5,0.001510,1.0
2,465,Musical Theater,4,20,Spoken,1245,4,0.003213,1.0,20,Spoken,1245,4,0.003213,1.0
3,189,Talk Radio,13,65,Radio,367,13,0.035422,1.0,20,Spoken,1245,13,0.010442,1.0
4,1032,Turkish,10,102,Middle East,58,10,0.172414,1.0,2,International,3311,10,0.003020,1.0
5,374,Banter,5,20,Spoken,1245,5,0.004016,1.0,20,Spoken,1245,5,0.004016,1.0
6,173,N. Indian Traditional,3,86,Indian,108,3,0.027778,1.0,2,International,3311,3,0.000906,1.0
7,493,Western Swing,1,651,Country & Western,41,1,0.024390,1.0,9,Country,1443,1,0.000693,1.0
8,377,Deep Funk,1,19,Funk,566,1,0.001767,1.0,14,Soul-RnB,1105,1,0.000905,1.0
9,808,Salsa,1,46,Latin America,351,1,0.002849,1.0,2,International,3311,1,0.000302,1.0


In [8]:
# ============================================================
# 8. MERGE ROUTER + CO-OCCURRENCE INTO FINAL ROUTING TABLE
# ============================================================

rare_tail_routing_table_df = rare_tail_router_df.merge(
    cooccurrence_df,
    on=["rare_tail_genre_id", "rare_tail_genre_name", "training_count", "anchor_candidate_id",
        "anchor_candidate_name", "root_candidate_id", "root_candidate_name"],
    how="left"
)

print("Final rare-tail routing table:")
display(rare_tail_routing_table_df)

Final rare-tail routing table:


,rare_tail_genre_id,rare_tail_genre_name,parent_id,parent_name,root_genre_id,root_genre_name,training_count,validation_count,test_count,total_count,...,root_candidate_name,fallback_mode,anchor_train_count,cooccur_anchor_count,p_rare_given_anchor,p_anchor_given_rare,root_train_count,cooccur_root_count,p_rare_given_root,p_root_given_rare
0,176,Pacific,2,International,2,International,17,2,4,23,...,International,Hierarchy-triggered fallback,3311,17,0.005134,1.0,3311,17,0.005134,1.0
1,1060,Tango,46,Latin America,2,International,5,6,12,23,...,International,Hierarchy-triggered fallback,351,5,0.014245,1.0,3311,5,0.001510,1.0
2,465,Musical Theater,20,Spoken,20,Spoken,4,4,10,18,...,Spoken,Hierarchy-triggered fallback,1245,4,0.003213,1.0,1245,4,0.003213,1.0
3,189,Talk Radio,65,Radio,20,Spoken,13,1,1,15,...,Spoken,Hierarchy-triggered fallback,367,13,0.035422,1.0,1245,13,0.010442,1.0
4,1032,Turkish,102,Middle East,2,International,10,0,5,15,...,International,Hierarchy-triggered fallback,58,10,0.172414,1.0,3311,10,0.003020,1.0
5,374,Banter,20,Spoken,20,Spoken,5,0,0,5,...,Spoken,Hierarchy-triggered fallback,1245,5,0.004016,1.0,1245,5,0.004016,1.0
6,173,N. Indian Traditional,86,Indian,2,International,3,0,1,4,...,International,Hierarchy-triggered fallback,108,3,0.027778,1.0,3311,3,0.000906,1.0
7,493,Western Swing,651,Country & Western,9,Country,1,1,2,4,...,Country,Hierarchy-triggered fallback,41,1,0.024390,1.0,1443,1,0.000693,1.0
8,377,Deep Funk,19,Funk,14,Soul-RnB,1,0,0,1,...,Soul-RnB,Hierarchy-triggered fallback,566,1,0.001767,1.0,1105,1,0.000905,1.0
9,808,Salsa,46,Latin America,2,International,1,0,0,1,...,International,Hierarchy-triggered fallback,351,1,0.002849,1.0,3311,1,0.000302,1.0


In [9]:
# ============================================================
# 9. BUILD ANCHOR -> RARE-TAIL SUGGESTION TABLE
# ============================================================

anchor_to_tail_df = rare_tail_routing_table_df.copy()

anchor_to_tail_df["anchor_score"] = anchor_to_tail_df["p_rare_given_anchor"].fillna(-1)
anchor_to_tail_df["root_score"] = anchor_to_tail_df["p_rare_given_root"].fillna(-1)

anchor_to_tail_df = anchor_to_tail_df.sort_values(
    ["anchor_candidate_name", "anchor_score", "training_count", "rare_tail_genre_name"],
    ascending=[True, False, False, True]
).reset_index(drop=True)

anchor_to_tail_display_df = anchor_to_tail_df[
    [
        "anchor_candidate_id",
        "anchor_candidate_name",
        "rare_tail_genre_id",
        "rare_tail_genre_name",
        "training_count",
        "validation_count",
        "test_count",
        "p_rare_given_anchor",
        "p_anchor_given_rare",
        "fallback_mode"
    ]
].copy()

print("Anchor to rare-tail suggestion table:")
display(anchor_to_tail_display_df)

Anchor to rare-tail suggestion table:


,anchor_candidate_id,anchor_candidate_name,rare_tail_genre_id,rare_tail_genre_name,training_count,validation_count,test_count,p_rare_given_anchor,p_anchor_given_rare,fallback_mode
0,651,Country & Western,493,Western Swing,1,1,2,0.024390,1.0,Hierarchy-triggered fallback
1,19,Funk,377,Deep Funk,1,0,0,0.001767,1.0,Hierarchy-triggered fallback
2,86,Indian,173,N. Indian Traditional,3,0,1,0.027778,1.0,Hierarchy-triggered fallback
3,86,Indian,175,Bollywood,0,0,0,0.000000,NaN,Inventory only
4,86,Indian,174,South Indian Traditional,0,1,14,0.000000,NaN,Inventory only
5,2,International,176,Pacific,17,2,4,0.005134,1.0,Hierarchy-triggered fallback
6,4,Jazz,178,Be-Bop,0,0,0,0.000000,NaN,Inventory only
7,46,Latin America,1060,Tango,5,6,12,0.014245,1.0,Hierarchy-triggered fallback
8,46,Latin America,808,Salsa,1,0,0,0.002849,1.0,Hierarchy-triggered fallback
9,102,Middle East,1032,Turkish,10,0,5,0.172414,1.0,Hierarchy-triggered fallback


In [10]:
# ============================================================
# 10. ROUTING COVERAGE CHECK USING GROUND-TRUTH TRACK LABELS
# ============================================================

full_master_indexed = full_master_df.set_index("track_id", drop=False)

coverage_rows = []

for _, row in rare_tail_routing_table_df.iterrows():
    rare_id = int(row["rare_tail_genre_id"])
    anchor_id = row["anchor_candidate_id"]
    root_id = row["root_candidate_id"]

    rare_pairs = rare_tail_pairs_df[rare_tail_pairs_df["rare_tail_genre_id"] == rare_id].copy()
    track_ids = rare_pairs["track_id"].astype(int).tolist()

    anchor_truth_hits = 0
    root_truth_hits = 0

    for tid in track_ids:
        track_row = full_master_indexed.loc[tid]

        if anchor_id is not None and f"genre_{anchor_id}" in full_master_indexed.columns:
            anchor_truth_hits += int(track_row[f"genre_{anchor_id}"] == 1)

        if root_id is not None and f"genre_{root_id}" in full_master_indexed.columns:
            root_truth_hits += int(track_row[f"genre_{root_id}"] == 1)

    total_pairs = len(track_ids)

    coverage_rows.append({
        "rare_tail_genre_id": rare_id,
        "rare_tail_genre_name": row["rare_tail_genre_name"],
        "total_pairs": total_pairs,
        "anchor_candidate_id": anchor_id,
        "anchor_candidate_name": row["anchor_candidate_name"],
        "anchor_truth_hits": anchor_truth_hits,
        "anchor_truth_coverage": anchor_truth_hits / total_pairs if total_pairs > 0 else np.nan,
        "root_candidate_id": root_id,
        "root_candidate_name": row["root_candidate_name"],
        "root_truth_hits": root_truth_hits,
        "root_truth_coverage": root_truth_hits / total_pairs if total_pairs > 0 else np.nan
    })

routing_coverage_df = pd.DataFrame(coverage_rows).sort_values(
    ["anchor_truth_coverage", "root_truth_coverage", "total_pairs"],
    ascending=[False, False, False]
).reset_index(drop=True)

print("Rare-tail routing coverage table:")
display(routing_coverage_df)

Rare-tail routing coverage table:


,rare_tail_genre_id,rare_tail_genre_name,total_pairs,anchor_candidate_id,anchor_candidate_name,anchor_truth_hits,anchor_truth_coverage,root_candidate_id,root_candidate_name,root_truth_hits,root_truth_coverage
0,176,Pacific,23,2,International,23,1.0,2,International,23,1.0
1,1060,Tango,23,46,Latin America,23,1.0,2,International,23,1.0
2,465,Musical Theater,18,20,Spoken,18,1.0,20,Spoken,18,1.0
3,189,Talk Radio,15,65,Radio,15,1.0,20,Spoken,15,1.0
4,1032,Turkish,15,102,Middle East,15,1.0,2,International,15,1.0
5,174,South Indian Traditional,15,86,Indian,15,1.0,2,International,15,1.0
6,374,Banter,5,20,Spoken,5,1.0,20,Spoken,5,1.0
7,173,N. Indian Traditional,4,86,Indian,4,1.0,2,International,4,1.0
8,493,Western Swing,4,651,Country & Western,4,1.0,9,Country,4,1.0
9,377,Deep Funk,1,19,Funk,1,1.0,14,Soul-RnB,1,1.0


In [11]:
# ============================================================
# 11. BUILD STAGE-2 EXECUTION STRATEGY TABLE
# ============================================================

stage2_strategy_rows = []

for _, row in rare_tail_routing_table_df.iterrows():
    gid = int(row["rare_tail_genre_id"])
    status = row["feasibility_status"]
    mode = row["fallback_mode"]

    if mode == "Inventory only":
        execution_rule = (
            "Do not predict directly. Keep in inventory and expose only as taxonomy metadata or manual review label."
        )
    else:
        execution_rule = (
            "Only surface as a fallback suggestion when the anchor/root candidate family is triggered by Stage 1."
        )

    if status in ["Very scarce", "Extremely scarce"]:
        recommendation = "Rare-tail hint only"
    elif status == "Partial split support":
        recommendation = "Hierarchy fallback only"
    elif status == "Train only":
        recommendation = "Training-only tail; no reliable evaluation support"
    elif status == "No train support":
        recommendation = "Inventory only"
    else:
        recommendation = "Review manually"

    stage2_strategy_rows.append({
        "rare_tail_genre_id": gid,
        "rare_tail_genre_name": row["rare_tail_genre_name"],
        "anchor_candidate_name": row["anchor_candidate_name"],
        "root_candidate_name": row["root_candidate_name"],
        "feasibility_status": status,
        "fallback_mode": mode,
        "execution_rule": execution_rule,
        "recommendation": recommendation
    })

stage2_strategy_df = pd.DataFrame(stage2_strategy_rows)

print("Stage-2 execution strategy:")
display(stage2_strategy_df)

Stage-2 execution strategy:


,rare_tail_genre_id,rare_tail_genre_name,anchor_candidate_name,root_candidate_name,feasibility_status,fallback_mode,execution_rule,recommendation
0,176,Pacific,International,International,Very scarce,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Rare-tail hint only
1,1060,Tango,Latin America,International,Extremely scarce,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Rare-tail hint only
2,465,Musical Theater,Spoken,Spoken,Extremely scarce,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Rare-tail hint only
3,189,Talk Radio,Radio,Spoken,Very scarce,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Rare-tail hint only
4,1032,Turkish,Middle East,International,Partial split support,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Hierarchy fallback only
5,374,Banter,Spoken,Spoken,Train only,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Training-only tail; no reliable evaluation sup...
6,173,N. Indian Traditional,Indian,International,Partial split support,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Hierarchy fallback only
7,493,Western Swing,Country & Western,Country,Extremely scarce,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Rare-tail hint only
8,377,Deep Funk,Funk,Soul-RnB,Train only,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Training-only tail; no reliable evaluation sup...
9,808,Salsa,Latin America,International,Train only,Hierarchy-triggered fallback,Only surface as a fallback suggestion when the...,Training-only tail; no reliable evaluation sup...


In [12]:
# ============================================================
# 12. BUILD FULL-161 READINESS SUMMARY
# ============================================================

num_inventory_only = int((stage2_strategy_df["fallback_mode"] == "Inventory only").sum())
num_hierarchy_fallback = int((stage2_strategy_df["fallback_mode"] == "Hierarchy-triggered fallback").sum())
num_with_anchor = int(stage2_strategy_df["anchor_candidate_name"].notna().sum())

readiness_rows = [
    {
        "Component": "Stage 1 candidate-150 benchmark model",
        "Status": "Ready",
        "Details": "Expanded hybrid benchmark system is already frozen."
    },
    {
        "Component": "Stage 2 rare-tail router",
        "Status": "Ready",
        "Details": f"Built routing table for {len(stage2_strategy_df)} rare-tail labels."
    },
    {
        "Component": "Rare-tail labels with candidate anchors",
        "Status": "Ready",
        "Details": f"{num_with_anchor} rare-tail labels map to a nearest candidate ancestor."
    },
    {
        "Component": "Hierarchy-triggered fallback labels",
        "Status": "Ready",
        "Details": f"{num_hierarchy_fallback} labels can be surfaced through hierarchy-based fallback."
    },
    {
        "Component": "Inventory-only rare-tail labels",
        "Status": "Ready",
        "Details": f"{num_inventory_only} labels should remain inventory-only until stronger support exists."
    },
    {
        "Component": "Full 161-label inference pipeline",
        "Status": "Next step",
        "Details": "Combine Stage 1 benchmark predictions with Stage 2 fallback routing in the next notebook."
    }
]

full161_readiness_df = pd.DataFrame(readiness_rows)

print("Full-161 readiness summary:")
display(full161_readiness_df)

Full-161 readiness summary:


,Component,Status,Details
0,Stage 1 candidate-150 benchmark model,Ready,Expanded hybrid benchmark system is already fr...
1,Stage 2 rare-tail router,Ready,Built routing table for 13 rare-tail labels.
2,Rare-tail labels with candidate anchors,Ready,13 rare-tail labels map to a nearest candidate...
3,Hierarchy-triggered fallback labels,Ready,10 labels can be surfaced through hierarchy-ba...
4,Inventory-only rare-tail labels,Ready,3 labels should remain inventory-only until st...
5,Full 161-label inference pipeline,Next step,Combine Stage 1 benchmark predictions with Sta...


In [13]:
# ============================================================
# 13. SAVE OUTPUTS
# ============================================================

os.makedirs("../data/processed", exist_ok=True)

rare_tail_routing_table_df.to_csv(
    "../data/processed/full161_rare_tail_routing_table.csv",
    index=False
)

anchor_to_tail_display_df.to_csv(
    "../data/processed/full161_anchor_to_rare_tail_table.csv",
    index=False
)

routing_coverage_df.to_csv(
    "../data/processed/full161_rare_tail_routing_coverage.csv",
    index=False
)

stage2_strategy_df.to_csv(
    "../data/processed/full161_stage2_execution_strategy.csv",
    index=False
)

full161_readiness_df.to_csv(
    "../data/processed/full161_readiness_summary.csv",
    index=False
)

print("Saved full-161 rare-tail fallback router outputs.")

Saved full-161 rare-tail fallback router outputs.


In [14]:
# ============================================================
# 14. INTERPRETATION NOTES
# ============================================================

print("1. The rare-tail stage should be handled through hierarchy-triggered fallback, not direct conventional training.")
print("2. This notebook maps each rare-tail label to a candidate ancestor and computes fallback statistics.")
print("3. The outputs now define how Stage 2 should behave in a full 161-label pipeline.")
print("4. The next notebook should combine the frozen candidate-150 benchmark model with this rare-tail router.")
print("5. After that, the project will have a working full-genre inference pipeline design.")

1. The rare-tail stage should be handled through hierarchy-triggered fallback, not direct conventional training.
2. This notebook maps each rare-tail label to a candidate ancestor and computes fallback statistics.
3. The outputs now define how Stage 2 should behave in a full 161-label pipeline.
4. The next notebook should combine the frozen candidate-150 benchmark model with this rare-tail router.
5. After that, the project will have a working full-genre inference pipeline design.
